# 🚗 YOLOv11 Fine-Tuning — Singapore Traffic Detection

Fine-tunes YOLOv11n on **UA-DETRAC** traffic dataset.

**Platform:** Kaggle (T4 GPU, 12h sessions) or Colab (T4, shorter sessions)

**Output:** `best.pt` model for 6-class traffic detection

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics roboflow mlflow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# 2. Download UA-DETRAC dataset via Roboflow
# Get free API key at https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = ''  # <-- PASTE YOUR KEY HERE

if not ROBOFLOW_API_KEY:
    raise ValueError('Set your ROBOFLOW_API_KEY above! Get one free at https://app.roboflow.com/settings/api')

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace().project('ua-detrac-yvkna')
version = project.version(1)
dataset = version.download('yolov11')

print(f'\n✅ Dataset downloaded to: {dataset.location}')

In [ ]:
# 3. Verify dataset structure
import os
from pathlib import Path

dataset_path = Path(dataset.location)
for split in ['train', 'valid', 'test']:
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob('*')))
        n_lbls = len(list(lbl_dir.glob('*')))
        print(f'{split}: {n_imgs} images, {n_lbls} labels')

# Show data.yaml
yaml_path = dataset_path / 'data.yaml'
print(f'\n--- data.yaml ---')
print(yaml_path.read_text())

In [ ]:
# 4. Train YOLOv11n
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # Start from pretrained

results = model.train(
    data=str(dataset_path / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # Performance
    workers=4,
    amp=True,
    cache=True,
    project='runs',
    name='sg_traffic_yolo11n',
    verbose=True,
)

print(f'\n✅ Training complete!')
print(f'Best model: {results.save_dir}/weights/best.pt')

In [ ]:
# 5. Evaluate on test set
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val(data=str(dataset_path / 'data.yaml'), split='test')

print(f'\n📊 Test Results:')
print(f'   mAP50:     {metrics.box.map50:.4f}')
print(f'   mAP50-95:  {metrics.box.map:.4f}')
print(f'   Precision:  {metrics.box.mp:.4f}')
print(f'   Recall:     {metrics.box.mr:.4f}')

In [ ]:
# 6. Export to ONNX for fast inference
best_model.export(format='onnx', dynamic=True, simplify=True)
print('✅ ONNX export complete')

In [ ]:
# 7. Save artifacts
import shutil

# For Colab: save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = '/content/drive/MyDrive/sg_smart_city/models'
except ImportError:
    # Kaggle: save to /kaggle/working (auto-downloaded)
    save_dir = '/kaggle/working/models'

os.makedirs(save_dir, exist_ok=True)
shutil.copy(f'{results.save_dir}/weights/best.pt', f'{save_dir}/best.pt')
shutil.copy(f'{results.save_dir}/weights/best.onnx', f'{save_dir}/best.onnx')

# Copy training curves
for f in ['results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png']:
    src = f'{results.save_dir}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{save_dir}/{f}')

print(f'\n✅ All artifacts saved to: {save_dir}')
print(f'   best.pt    — PyTorch model')
print(f'   best.onnx  — ONNX export')
print(f'   results.csv — training metrics')